In [1]:
#!pip install bert-score transformers torch sentence-transformers numpy spacy nltk ollama accelerate rapidfuzz

In [2]:
import os
import re
import ast
import nltk
import json
import torch
import spacy
import ollama
import numpy as np
import unicodedata
import torch.nn.functional as F
from rapidfuzz import fuzz
from bert_score import score
from transformers import logging, pipeline

from sentence_transformers import SentenceTransformer, util
from transformers import BartTokenizer, BartForConditionalGeneration, AutoTokenizer,AutoModelForCausalLM, AutoModelForSequenceClassification


/home/jupyter-ter-acct-1/.local/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [ ]:
proxy = "http://cache.ha.aaaa-bbbbb.fr:xxxx/"

os.environ['http_proxy'] = proxy
os.environ['https_proxy'] = proxy
os.environ['HTTP_PROXY'] = proxy
os.environ['HTTPS_PROXY'] = proxy

In [5]:
logging.set_verbosity_error()

## Contextual Semantic Metrics 

In [6]:
def calculate_bertscore(paragraph:str, rewrite:str, type="bert"):
    if type == "bert":
        model = "bert-base-uncased"
    if type == "scibert":
        model = "allenai/scibert_scivocab_uncased"

    P, R, F1 = score(
    cands= rewrite,
    refs= paragraph,
    lang="en",
    model_type=model,
    verbose=False,
    )
    
    return F1.mean().item()

In [ ]:
def calculate_bartscore(paragraph:str, rewrite:str):
    model_name = "facebook/bart-large"
    tokenizer = BartTokenizer.from_pretrained(model_name)
    model = BartForConditionalGeneration.from_pretrained(model_name)
    model.eval()

    with torch.no_grad():
        inputs = tokenizer(paragraph, return_tensors="pt", truncation=True)
        labels = tokenizer(rewrite, return_tensors="pt", truncation=True)["input_ids"]

        outputs = model(**inputs, labels=labels)
        loss = outputs.loss  
        
        bart_score = -loss.item()

    return bart_score

In [8]:
def calculate_cosine_similarity(paragraph:str, rewrite:str):
    model_name = "sentence-transformers/all-mpnet-base-v2"
    model = SentenceTransformer(model_name)
    
    embeddings = model.encode([paragraph, rewrite], show_progress_bar=False)
    
    vec1 = embeddings[0]
    vec2 = embeddings[1]
    
    cosine = np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )
    return float(cosine)

In [9]:
def calculate_semantic_metrics(paragraph, rewrite):
    bert_score = calculate_bertscore(paragraph, rewrite)
    #scibert_score = calculate_bertscore(paragraph, rewrite, "scibert")
    bart_score = calculate_bartscore(paragraph, rewrite)
    cosinus_similarity = calculate_cosine_similarity(paragraph[0], rewrite[0])
    return {"BertScore":bert_score,
            #"SciBertScore": scibert_score,
            "BartScore": bart_score,
            "Cos_Sim":cosinus_similarity}

## Factual metrics

In [10]:
factcc_tokenizer = AutoTokenizer.from_pretrained("manueldeprada/FactCC")
factcc_model = AutoModelForSequenceClassification.from_pretrained("manueldeprada/FactCC")

def factcc_score(paragraph, rewrite):

    inputs = factcc_tokenizer(
        paragraph,
        rewrite,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = factcc_model(**inputs)
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=1)

    return prediction.squeeze().item()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [11]:
#spacy.cli.download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")

In [12]:
def factacc_score(paragraph, rewrite):

    doc = nlp(rewrite)

    facts = []
    correct = 0

    for token in doc:
        if token.dep_ == "nsubj":
            subject = token.text
            verb = token.head.text

            for child in token.head.children:
                if child.dep_ == "dobj":
                    obj = child.text
                    facts.append((subject, verb, obj))

    for fact in facts:
        if all(word.lower() in paragraph.lower() for word in fact):
            correct += 1

    if len(facts) == 0:
        return 0

    return correct / len(facts)

In [ ]:
def dae_score(paragraph, rewrite):

    doc_src = nlp(paragraph)
    doc_rep = nlp(rewrite)

    src_arcs = set(
        (t.head.lemma_.lower(), t.dep_, t.lemma_.lower())
        for t in doc_src
    )

    rep_arcs = [
        (t.head.lemma_.lower(), t.dep_, t.lemma_.lower())
        for t in doc_rep
    ]

    if not rep_arcs:
        return 0.0

    supported = 0

    for arc in rep_arcs:
        if arc in src_arcs:
            supported += 1

    return supported / len(rep_arcs)

In [ ]:
model_path = "/home/partage/Llama-3.2-1B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(model_path)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.float16,
)

def llm(prompt):

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            pad_token_id=llm_tokenizer.eos_token_id,
            max_new_tokens=300,
            temperature=0.0,
            do_sample=False
        )

    return llm_tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [15]:
def extract_facts(text):

    prompt = f"""
You are a strict information extraction system.

Extract ONLY atomic factual statements.

RULES:
- Return ONLY a valid Python list of strings.
- Do NOT include any code, function, variable name, or explanation.
- Do NOT wrap the result in markdown.
- Do NOT add any text before or after the list.
- Each element must be a short factual sentence.
- If no facts are found, return [].

EXAMPLE:
["Fact 1", "Fact 2"]

TEXT:
{text}

OUTPUT:
"""

    response = llm(prompt)
    return response.split("Extract atomic factual statements")[-1].strip()

In [16]:
def parse_facts(response):
    try:
        # trouve toutes les listes dans le texte
        matches = re.findall(r"\[.*?\]", response, re.DOTALL)

        if matches:
            # prend la dernière (souvent la bonne réponse)
            return ast.literal_eval(matches[-1])

    except:
        pass

    return []

In [ ]:
def safe_str(x):
    if x is None:
        return ""
    return str(x)
    
def naive_fact_score(source_facts, summary_facts):

    if not summary_facts:
        return 0.0

    # sécurisation totale des types
    source_facts = [safe_str(s) for s in source_facts]
    summary_facts = [safe_str(s) for s in summary_facts]

    supported = 0

    for fact in summary_facts:
        fact = fact.strip()
        if not fact:
            continue

        if any(
            fuzz.partial_ratio(fact.lower(), s.lower()) > 85
            for s in source_facts
        ):
            supported += 1

    return supported / len(summary_facts)

In [18]:
# modèle NLI
nli = pipeline("text-classification", model="facebook/bart-large-mnli")

def classify_nli(premise, hypothesis):

    result = nli(f"{premise} </s> {hypothesis}")[0]

    return result["label"].upper(), result["score"]


def nli_fact_based_score(source_facts, summary_facts):

    if not summary_facts:
        return 0.0

    supported = 0
    contradicted = 0

    for h in summary_facts:

        best_entail = 0
        best_contra = 0

        for p in source_facts:

            label, score = classify_nli(p, h)

            if label == "ENTAILMENT":
                best_entail = max(best_entail, score)

            elif label == "CONTRADICTION":
                best_contra = max(best_contra, score)

        # décision finale robuste
        if best_entail > best_contra and best_entail > 0.5:
            supported += 1

        elif best_contra > best_entail and best_contra > 0.5:
            contradicted += 1

    total = len(summary_facts)

    return supported / total - contradicted / total


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [19]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

def cos_sim_fact_based_score(source_facts, summary_facts):

    supported = 0

    for fact in summary_facts:
        fact_emb = embedding_model.encode(fact, convert_to_tensor=True)

        for s in source_facts:
            s_emb = embedding_model.encode(s, convert_to_tensor=True)
            sim = util.cos_sim(fact_emb, s_emb)

            if sim > 0.75:
                supported += 1
                break

    return supported / len(summary_facts) if summary_facts else 0

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [20]:
def clean_fact_list(facts):
    cleaned = []
    for f in facts:
        if isinstance(f, str):
            cleaned.append(f)
        elif isinstance(f, (int, float)):
            cleaned.append(str(f))
        elif isinstance(f, list):
            cleaned.extend([str(x) for x in f])
    return cleaned

In [21]:
# -----------------------------
# FactScore
# -----------------------------
def factscore(source, summary):
    rew1 = extract_facts(source)
    source = parse_facts(rew1)
    rew2 = extract_facts(summary)
    summary = parse_facts(rew2)

    source_facts = clean_fact_list(source)
    summary_facts = clean_fact_list(summary)

    naive_score = naive_fact_score(source_facts, summary_facts)
    cos_sim_score = cos_sim_fact_based_score(source_facts, summary_facts)
    nli_score = nli_fact_based_score(source_facts, summary_facts)

    final_score = (
        0.2 * naive_score +
        0.2 * cos_sim_score +
        0.6 * nli_score
    )

    print(f"Naive: {naive_score} | Cosine: {cos_sim_score} | NLI: {nli_score}")

    return final_score

In [22]:
# def calculate_factual_metrics(paragraph, rewrite):
#     fact_cc = factcc_score(paragraph, rewrite)
#     fact_acc = factacc_score(paragraph, rewrite)
#     dae = dae_score(paragraph, rewrite)
#     fact_score = factscore(paragraph, rewrite)

#     return {"FactCC":fact_cc,
#             "FactAcc": fact_acc,
#             "DAE":fact_score,
#             "FActScore":fact_score}

def calculate_factual_metrics(paragraph, rewrite):

    fact_cc = factcc_score(paragraph, rewrite)
    fact_acc = factacc_score(paragraph, rewrite)
    dae = dae_score(paragraph, rewrite)
    fact_score = factscore(paragraph, rewrite)

    return {
        "FactCC": fact_cc,
        "FactAcc": fact_acc,
        "DAE": dae,
        "FactScore": fact_score
    }

In [23]:
def keys_update(dico,index=1):
    new_dico = {f"{k}{index}": v for k, v in dico.items()}
    return new_dico

In [24]:
# def clean(text):
#     if text is None:
#         return ""
#     return unicodedata.normalize("NFKC", str(text)).replace("’", "'").replace("“", '"').replace("”", '"')

In [25]:

def compute_all_metrics(data):

    for article in data:

        paragraph = article.get("Paragraphes", "")

        for i in range(1, 4):

            rewrite = article.get(f"Rew{i}", "")

            semantic_metrics = calculate_semantic_metrics(
                [paragraph], [rewrite]
            )
            article.update(keys_update(semantic_metrics, i))

            factual_metrics = calculate_factual_metrics(
                paragraph,
                rewrite
            )
            article.update(keys_update(factual_metrics, i))

    return data


## Code Tests

In [26]:
with open("../data/json/rewrites.json","r" , encoding='utf-8') as file:
    data = json.load(file)

In [27]:
all_metrics = compute_all_metrics(data)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Naive: 0.0 | Cosine: 1.0 | NLI: 0.3333333333333333


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Naive: 1.0 | Cosine: 0.75 | NLI: -0.75


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Naive: 0.0 | Cosine: 0.0 | NLI: 0.0


In [28]:
filename = "../data/json/metrics.json"
if not os.path.exists(filename):
    with open(filename, "w", encoding="utf-8") as f:
        pass

In [29]:
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(all_metrics, f, indent=4, ensure_ascii=False)